Статический парсинг вакансий, связанных с уходом за животными, на странице тг канала rabota_moskva.

In [ ]:
import re
import time
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [ ]:
class TelegramVacancyParser:
    def __init__(self, channel_name, delay = 1.0):
        self.channel_name = channel_name
        self.base_url = f"https://t.me/s/{channel_name}"
        self.delay = delay

        self.headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0 Safari/537.36"
            )
        }

        self.pet_keywords = re.compile(
            r"ветеринар|ветклиник|вет\.?клиник|грумер|зоосалон|"
            r"зоонян|котоотел|отель для кош|уход.*кош|уход.*животн|"
            r"собак|кошк|животн|питомц",
            re.IGNORECASE
        )

        self.vacancy_keywords = re.compile(
            r"требуется|ищем|ищет|нужен|нужна|приглашаем|"
            r"вакансия|з/п|зарплата|оклад|оплата|график",
            re.IGNORECASE
        )

        self.bad_keywords = re.compile(
            r"домработниц|семейная пара|помощниц[аы] по хозяйству|"
            r"агроном|декоративных растений|питомник декоративных растений|"
            r"сельскохозяйственных животных|кормовых добавок|"
            r"оператор call-центра|call-центр|"
            r"сбор денег|пожертвован|помощь приюту|"
            r"редактор|маркетолог|продажам кормовых добавок",
            re.IGNORECASE
        )

    def fetch_page(self, query = None, before = None):
        params = {}

        if query:
            params["q"] = query

        if before:
            params["before"] = before

        response = requests.get(self.base_url, params=params, headers=self.headers, timeout=20)

        response.raise_for_status()

        return BeautifulSoup(response.text, "lxml")

    def get_next_before(self, soup):
        more_link = soup.find("a", class_="tme_messages_more")

        if not more_link:
            return None

        href = more_link.get("href", "")

        match = re.search(r"before=(\d+)", href)

        if match:
            return match.group(1)

        return None

    def normalize_text(self, text):
        text = text.replace("\xa0", " ")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()

    def extract_messages(self, soup, query = None):
        rows = []

        messages = soup.find_all("div", class_="tgme_widget_message")

        for message in messages:
            text_block = message.find("div", class_="tgme_widget_message_text")

            if not text_block:
                continue

            raw_text = text_block.get_text("\n", strip=True)
            raw_text = self.normalize_text(raw_text)

            if not raw_text:
                continue

            link_tag = message.find("a", class_="tgme_widget_message_date")
            post_url = urljoin("https://t.me", link_tag.get("href")) if link_tag else None

            rows.append({"post_url": post_url, "raw_text": raw_text, "search_query": query})

        return rows

    def is_relevant_pet_vacancy(self, text):
        if not self.pet_keywords.search(text):
            return False

        if not self.vacancy_keywords.search(text):
            return False

        if self.bad_keywords.search(text):
            return False

        return True

    def normalize_position(self, text):
        text_lower = text.lower()

        if "ветеринарный врач" in text_lower or "ветврач" in text_lower:
            return "Ветеринарный врач"

        if "ассистент" in text_lower and "ветеринар" in text_lower:
            return "Ассистент ветеринарного врача"

        if "грумер" in text_lower:
            return "Грумер"

        if "администратор" in text_lower and ("ветеринар" in text_lower or "ветклиник" in text_lower):
            return "Администратор ветеринарной клиники"

        if "администратор" in text_lower and "зоосалон" in text_lower:
            return "Администратор зоосалона"

        if (
            "зооняня" in text_lower
            or "догситтер" in text_lower
            or "отель для собак" in text_lower
            or "уходу за собаками" in text_lower
            or "уход за собаками" in text_lower
            or "котоотель" in text_lower
            or "отель для кош" in text_lower
            or "уходу за кошками" in text_lower
            or "уход за кошками" in text_lower
        ):
            return "Зооняня"

        if "уходу за животными" in text_lower or "уход за животными" in text_lower:
            return "Сотрудник по уходу за животными"

        return None

    def money_to_rub(self, number, unit):
        number = number.replace(" ", "").replace("\xa0", "")

        if re.fullmatch(r"\d{1,3}(?:\.\d{3})+", number):
            number = number.replace(".", "")

        value = float(number.replace(",", "."))

        if unit:
            unit = unit.lower()

            if "тыс" in unit or "тысяч" in unit or unit in ["к", "k"]:
                value *= 1000

        return int(value)

    def extract_salary(self, text):
        salary_data = {"salary_text": None, "salary_min_rub": None, "salary_max_rub": None}

        text_for_search = re.sub(r"\b\d{1,2}[:.]\d{2}\b", "", text)

        number = r"\d{1,3}(?:[\s\xa0.]?\d{3})+|\d+(?:[,.]\d+)?"
        unit = r"тыс\.?|тысяч|к|k"

        candidate_lines = []

        for line in text_for_search.split("\n"):
            if re.search(r"з/п|зарплат|оклад|оплат|доход|ставка|₽|руб|р\b", line, re.IGNORECASE):
                candidate_lines.append(line)

        if not candidate_lines:
            return salary_data

        salary_text_source = "\n".join(candidate_lines)

        range_pattern = re.compile(
            rf"(?P<prefix>от)?\s*"
            rf"(?P<num1>{number})\s*"
            rf"(?P<unit1>{unit})?\s*"
            rf"(?:₽|руб\.?|рублей|рубля|р\b)?\s*"
            rf"(?:[-–—]|\s+до\s+)\s*"
            rf"(?P<num2>{number})\s*"
            rf"(?P<unit2>{unit})?\s*"
            rf"(?:₽|руб\.?|рублей|рубля|р\b)?",
            re.IGNORECASE
        )

        range_match = range_pattern.search(salary_text_source)

        if range_match:
            unit1 = range_match.group("unit1") or range_match.group("unit2")
            unit2 = range_match.group("unit2") or range_match.group("unit1")

            value1 = self.money_to_rub(range_match.group("num1"), unit1)
            value2 = self.money_to_rub(range_match.group("num2"), unit2)

            if max(value1, value2) < 1000:
                return salary_data

            salary_data["salary_text"] = range_match.group(0).strip()
            salary_data["salary_min_rub"] = min(value1, value2)
            salary_data["salary_max_rub"] = max(value1, value2)

            return salary_data

        single_pattern = re.compile(
            rf"(?P<prefix>от|до)?\s*"
            rf"(?P<num1>{number})\s*"
            rf"(?P<unit1>{unit})?\s*"
            rf"(?:₽|руб\.?|рублей|рубля|р\b)?",
            re.IGNORECASE
        )

        matches = list(single_pattern.finditer(salary_text_source))

        for match in matches:
            value = self.money_to_rub(match.group("num1"), match.group("unit1"))

            if value < 1000:
                continue

            salary_data["salary_text"] = match.group(0).strip()

            prefix = match.group("prefix")

            if prefix and prefix.lower() == "от":
                salary_data["salary_min_rub"] = value

            elif prefix and prefix.lower() == "до":
                salary_data["salary_max_rub"] = value

            else:
                salary_data["salary_min_rub"] = value
                salary_data["salary_max_rub"] = value

            return salary_data

        return salary_data

    def parse_record(self, row):
        text = row["raw_text"]

        salary = self.extract_salary(text)

        parsed = {
            **row,
            "position": self.normalize_position(text),
            "salary_text": salary["salary_text"],
            "salary_min_rub": salary["salary_min_rub"],
            "salary_max_rub": salary["salary_max_rub"]
            }

        return parsed

    def collect_by_queries(self, queries, pages_per_query = 5):
        all_rows = []

        for query in queries:
            before = None

            for page_num in range(pages_per_query):
                soup = self.fetch_page(query=query, before=before)

                rows = self.extract_messages(soup, query=query)
                all_rows.extend(rows)

                before = self.get_next_before(soup)

                if not before:
                    break

                time.sleep(self.delay)

            time.sleep(self.delay)

        return self.build_dataframe(all_rows)


    def build_dataframe(self, rows):
        unique = {}

        for row in rows:
            key = row["post_url"] or row["raw_text"][:100]
            unique[key] = row

        relevant_rows = [self.parse_record(row) for row in unique.values() if self.is_relevant_pet_vacancy(row["raw_text"])]

        df = pd.DataFrame(relevant_rows)

        if df.empty:
            return df

        df = df.dropna(subset=["position"])

        return df.reset_index(drop=True)




In [ ]:
queries = [
    "груминг",
    "грумер",
    "ветеринар",
    "ветклиника",
    "ассистент ветеринарного врача",
    "зоосалон",
    "зооняня",
    "администратор зоосалона",
    "администратор ветеринарной клиники",
    "догситтер",
    "котоотель",
    "отель для кош",
    "отель для собак",
    "уход за животными",
    "уход за собаками",
    "уход за кошками",
    "животные",
    "зоогостиниц"
]

parser = TelegramVacancyParser(channel_name="rabota_moskva", delay=1.0)

df = parser.collect_by_queries(queries=queries, pages_per_query=5)

df = df.drop_duplicates(subset=["position", "salary_text", "raw_text"], keep="first").reset_index(drop=True)

In [ ]:
display(df)

,post_url,raw_text,search_query,position,salary_text,salary_min_rub,salary_max_rub
0,https://t.me/rabota_moskva/443,Требуется\nгрумер\nв салон\nТребования:\n•Стаж...,грумер,Грумер,None,NaN,NaN
1,https://t.me/rabota_moskva/180,Грумер\nТребования: опыт работы с собаками раз...,уход за кошками,Грумер,None,NaN,NaN
2,https://t.me/rabota_moskva/4234,В\nзоосалон\n«Тесси» требуется грумер с опытом...,зоосалон,Грумер,от 80000,80000.0,NaN
3,https://t.me/rabota_moskva/12356,"Всем привет!\nС радостью сообщаем, что в нашей...",грумер,Грумер,80.000,80000.0,80000.0
4,https://t.me/rabota_moskva/14402,Грумер\n70 000 ₽\nОбязанности:\n• модельные ст...,животные,Грумер,70 000 ₽,70000.0,70000.0
5,https://t.me/rabota_moskva/5270,Ветеринарный консультант (м. Пражская)\nот 350...,ассистент ветеринарного врача,Ассистент ветеринарного врача,от 35000 до 49000 руб.,35000.0,49000.0
6,https://t.me/rabota_moskva/8970,Ассистент ветеринарного врача\nЗ/п от 45 000р ...,уход за животными,Ассистент ветеринарного врача,от 45 000р до 70 000р,45000.0,70000.0
7,https://t.me/rabota_moskva/12053,Администратор\nветеринарной\nклиники\n60 000 ₽...,администратор ветеринарной клиники,Администратор ветеринарной клиники,60 000 ₽,60000.0,60000.0
8,https://t.me/rabota_moskva/2283,Требуется\nадминистратор\nветеринарной\nклиник...,администратор ветеринарной клиники,Администратор ветеринарной клиники,от 2500 руб.,2500.0,NaN
9,https://t.me/rabota_moskva/11742,Ассистент\nветеринарного\nврача\n2500 ₽/смена\...,ассистент ветеринарного врача,Ассистент ветеринарного врача,2500 ₽,2500.0,2500.0
